<a href="https://colab.research.google.com/github/Siddesh-2004/FindYourPhone/blob/main/notebook/imageBasedRecommender.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import requests
import os
from PIL import Image
from io import BytesIO
from tqdm import tqdm

df = pd.read_csv("/content/phones.csv")

os.makedirs("phone_images", exist_ok=True)

failed = []

for idx, row in tqdm(df.iterrows(), total=len(df)):
    url = row["img"]
    save_path = f"phone_images/{idx}.jpg"

    if os.path.exists(save_path):
        continue

    try:
        response = requests.get(url, timeout=10)
        img = Image.open(BytesIO(response.content)).convert("RGB")
        img.save(save_path)
    except Exception as e:
        failed.append({"idx": idx, "url": url, "error": str(e)})

print(f"\nDownloaded: {len(df) - len(failed)}")
print(f"Failed: {len(failed)}")

if failed:
    pd.DataFrame(failed).to_csv("failed_downloads.csv", index=False)
    print("Failed URLs saved to failed_downloads.csv")

100%|██████████| 1020/1020 [09:23<00:00,  1.81it/s]


Downloaded: 1020
Failed: 0


In [ ]:
import numpy as np
import pandas as pd
import os
from PIL import Image
from tqdm import tqdm
import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input

# ── Load model (no top = no classification layer, just features) ───────────────
model = MobileNetV2(weights="imagenet", include_top=False, pooling="avg")

df = pd.read_csv("phones.csv")

embeddings = []
valid_indices = []

for idx in tqdm(range(len(df))):
    img_path = f"phone_images/{idx}.jpg"

    try:
        img = Image.open(img_path).convert("RGB").resize((224, 224))
        arr = np.array(img, dtype=np.float32)
        arr = preprocess_input(arr)
        arr = np.expand_dims(arr, axis=0)

        embedding = model.predict(arr, verbose=0).flatten()
        embeddings.append(embedding)
        valid_indices.append(idx)

    except Exception as e:
        print(f"Failed idx {idx}: {e}")

embeddings = np.array(embeddings)
valid_indices = np.array(valid_indices)

np.save("embeddings.npy", embeddings)
np.save("embedding_indices.npy", valid_indices)

print(f"Embeddings shape: {embeddings.shape}")
print(f"Valid phones: {len(valid_indices)}")

/tmp/ipykernel_4090/3824897382.py:11: UserWarning: `input_shape` is undefined or non-square, or `rows` is not in [96, 128, 160, 192, 224]. Weights for input shape (224, 224) will be loaded as the default.
  model = MobileNetV2(weights="imagenet", include_top=False, pooling="avg")


9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step


100%|██████████| 1020/1020 [02:11<00:00,  7.78it/s]

Embeddings shape: (1020, 1280)
Valid phones: 1020


In [ ]:
import numpy as np
import pandas as pd
from PIL import Image
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import normalize
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
from google.colab import files

# ── Load model + data ──────────────────────────────────────────────────────────
model = MobileNetV2(weights="imagenet", include_top=False, pooling="avg")
df = pd.read_csv("phones.csv")

# ── Zone weights ───────────────────────────────────────────────────────────────
ZONE_WEIGHTS = {
    "back": 0.70,
    "front_cam": 0.20,
    "display": 0.10
}


def crop_zones(img):
    """
    Split a phone image (front+back format) into 3 weighted zones.

    Layout:
    ┌──────────┬──────────┐
    │          │ front cam│
    │   back   │  (top-R) │
    │  + logo  ├──────────┤
    │  (left)  │ display  │
    │          │ (bot-R)  │
    └──────────┴──────────┘
    """
    w, h = img.size
    mid_x = w // 2
    mid_y = h // 2

    back     = img.crop((0,     0,     mid_x, h))       # left half
    front    = img.crop((mid_x, 0,     w,     mid_y))   # top-right
    display  = img.crop((mid_x, mid_y, w,     h))       # bottom-right

    return {"back": back, "front_cam": front, "display": display}


def zone_embedding(zone_img):
    zone_resized = zone_img.resize((224, 224))
    arr = np.array(zone_resized, dtype=np.float32)
    arr = preprocess_input(arr)
    arr = np.expand_dims(arr, axis=0)
    return model(arr, training=False).numpy().flatten()


def weighted_embedding(img_path):
    """Compute weighted combined embedding from all zones."""
    img = Image.open(img_path).convert("RGB")
    zones = crop_zones(img)

    combined = None
    for zone_name, zone_img in zones.items():
        emb = zone_embedding(zone_img)
        weight = ZONE_WEIGHTS[zone_name]
        if combined is None:
            combined = weight * emb
        else:
            combined += weight * emb

    return normalize(combined.reshape(1, -1)).flatten()


# ── Precompute weighted embeddings for all dataset images ──────────────────────
print("Computing weighted embeddings for dataset...")
dataset_embeddings = []
valid_indices = []

for idx in range(len(df)):
    img_path = f"phone_images/{idx}.jpg"
    try:
        emb = weighted_embedding(img_path)
        dataset_embeddings.append(emb)
        valid_indices.append(idx)
    except Exception as e:
        print(f"Failed idx {idx}: {e}")

dataset_embeddings = np.array(dataset_embeddings)
valid_indices = np.array(valid_indices)

np.save("embeddings_weighted.npy", dataset_embeddings)
np.save("embedding_indices_weighted.npy", valid_indices)
print(f"Done. Shape: {dataset_embeddings.shape}")


# ── Similarity search ──────────────────────────────────────────────────────────
def find_similar(img_path, top_n=10):
    query_emb = weighted_embedding(img_path).reshape(1, -1)
    scores = cosine_similarity(query_emb, dataset_embeddings)[0]

    top_indices = np.argsort(scores)[::-1][1:top_n+1]
    top_df_indices = valid_indices[top_indices]
    top_scores = scores[top_indices]

    results = df.iloc[top_df_indices][["name", "price"]].copy()
    results["similarity"] = (top_scores * 100).round(2)
    results["local_path"] = [f"phone_images/{i}.jpg" for i in top_df_indices]
    return results.reset_index(drop=True)


def show_results(results):
    fig, axes = plt.subplots(2, 5, figsize=(18, 8))
    axes = axes.flatten()
    fig.suptitle("Top 10 Visually Similar Phones", fontsize=14, fontweight="bold")

    for i, (_, row) in enumerate(results.iterrows()):
        img = mpimg.imread(row["local_path"])
        axes[i].imshow(img)
        axes[i].set_title(f"{row['name'][:22]}\n₹{row['price']}  {row['similarity']}%", fontsize=8)
        axes[i].axis("off")

    plt.tight_layout()
    plt.show()


# ── Upload and search ──────────────────────────────────────────────────────────
uploaded = files.upload()
query_path = list(uploaded.keys())[0]

results = find_similar(query_path, top_n=10)
print(results[["name", "price", "similarity"]].to_string(index=False))
show_results(results)

/tmp/ipykernel_3987/1168203815.py:14: UserWarning: `input_shape` is undefined or non-square, or `rows` is not in [96, 128, 160, 192, 224]. Weights for input shape (224, 224) will be loaded as the default.
  model = MobileNetV2(weights="imagenet", include_top=False, pooling="avg")


Computing weighted embeddings for dataset...
Done. Shape: (1020, 1280)
